# FMS Phase 4 Price Engine Visualization

Run this notebook in VS Code with a Python environment that has this project installed. It uses the real `PriceSimulationEngine` and renders an inline SVG chart through the Jupyter notebook renderer.

The chart shows time on the x axis and relative price movement on the y axis.

`Indexed to 100` means each stock starts at `100` on the chart, even though the real simulated prices are different. For example, AAPL might start near `$224` and NVDA might start near `$901`, but both begin at `100` so you can compare how their prices move over time.

In [1]:
from collections import defaultdict
from datetime import datetime, timedelta
from html import escape
from pathlib import Path
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pydantic": "pydantic",
    "tzdata": "tzdata",
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print(f"Installing missing notebook dependencies into: {sys.executable}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

from IPython.display import HTML, display

repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.simulation import (  # noqa: E402
    DEFAULT_SIMULATED_STOCKS,
    MARKET_TIMEZONE,
    PriceSimulationConfig,
    PriceSimulationEngine,
)

print(f"Using Python kernel: {sys.executable}")
print(f"Using repository: {repo_root}")

Using Python kernel: c:\Users\Administrator\Downloads\Github\FMS\.venv\Scripts\python.exe
Using repository: c:\Users\Administrator\Downloads\Github\FMS


In [2]:
def simulate_price_paths(seed=42, drift=0.08, steps=120, step_minutes=1):
    engine = PriceSimulationEngine(PriceSimulationConfig(seed=seed, drift=drift))
    start_time = datetime(2026, 8, 31, 8, 30, tzinfo=MARKET_TIMEZONE)
    points = engine.simulate(
        DEFAULT_SIMULATED_STOCKS,
        start_time=start_time,
        steps=steps,
        step_size=timedelta(minutes=step_minutes),
    )

    by_symbol = defaultdict(list)
    for point in points:
        by_symbol[point.symbol].append(point)

    series = []
    for symbol, symbol_points in by_symbol.items():
        first_price = float(symbol_points[0].price)
        values = [
            {
                "minute": index * step_minutes + step_minutes,
                "price": float(point.price),
                "indexed": float(point.price) / first_price * 100,
            }
            for index, point in enumerate(symbol_points)
        ]
        series.append({"symbol": symbol, "values": values})
    return series


series = simulate_price_paths(seed=42, drift=0.08, steps=120, step_minutes=1)

In [3]:
def render_price_paths(series, width=980, height=520):
    margin = {"top": 34, "right": 116, "bottom": 58, "left": 72}
    inner_width = width - margin["left"] - margin["right"]
    inner_height = height - margin["top"] - margin["bottom"]
    colors = [
        "#2563eb", "#dc2626", "#16a34a", "#9333ea", "#ca8a04",
        "#0891b2", "#db2777", "#4f46e5", "#65a30d", "#ea580c",
    ]

    all_values = [value for item in series for value in item["values"]]
    min_x = min(value["minute"] for value in all_values)
    max_x = max(value["minute"] for value in all_values)
    min_y = min(value["indexed"] for value in all_values)
    max_y = max(value["indexed"] for value in all_values)
    y_padding = max(0.2, (max_y - min_y) * 0.12)
    min_y -= y_padding
    max_y += y_padding

    def scale_x(value):
        return margin["left"] + ((value - min_x) / (max_x - min_x)) * inner_width

    def scale_y(value):
        return margin["top"] + (1 - ((value - min_y) / (max_y - min_y))) * inner_height

    x_ticks = [min_x, 30, 60, 90, max_x]
    y_step = (max_y - min_y) / 5
    y_ticks = [min_y + y_step * index for index in range(6)]

    x_axis_y = margin["top"] + inner_height
    y_axis_x = margin["left"]
    parts = [
        f'<svg viewBox="0 0 {width} {height}" width="100%" role="img" '
        'aria-labelledby="price-engine-title price-engine-desc" '
        'style="font-family: system-ui, -apple-system, Segoe UI, sans-serif; max-width: 100%;">',
        '<title id="price-engine-title">FMS simulated price paths</title>',
        '<desc id="price-engine-desc">Seeded geometric Brownian motion price paths for the ten default Phase 4 stocks.</desc>',
        f'<rect x="{margin["left"]}" y="{margin["top"]}" width="{inner_width}" height="{inner_height}" fill="white" stroke="#d1d5db" />',
        '<text x="20" y="24" font-size="18" font-weight="600" fill="#111827">FMS Phase 4 Price Engine</text>',
        '<text x="20" y="46" font-size="12" fill="#4b5563">Seed 42, 120 one-minute steps, each line starts at 100 for easier comparison.</text>',
    ]

    for tick in x_ticks:
        x = scale_x(tick)
        parts.append(f'<line x1="{x:.2f}" y1="{margin["top"]}" x2="{x:.2f}" y2="{x_axis_y}" stroke="#e5e7eb" />')
        parts.append(f'<text x="{x:.2f}" y="{x_axis_y + 22}" text-anchor="middle" font-size="11" fill="#374151">{tick:.0f}</text>')

    for tick in y_ticks:
        y = scale_y(tick)
        parts.append(f'<line x1="{margin["left"]}" y1="{y:.2f}" x2="{margin["left"] + inner_width}" y2="{y:.2f}" stroke="#e5e7eb" />')
        parts.append(f'<text x="{y_axis_x - 10}" y="{y + 4:.2f}" text-anchor="end" font-size="11" fill="#374151">{tick:.1f}</text>')

    parts.append(f'<text x="{margin["left"] + inner_width / 2}" y="{height - 14}" text-anchor="middle" font-size="12" fill="#111827">Time elapsed in simulation, minutes</text>')
    parts.append(f'<text transform="translate(18 {margin["top"] + inner_height / 2}) rotate(-90)" text-anchor="middle" font-size="12" fill="#111827">Relative price movement, start = 100</text>')

    for index, item in enumerate(series):
        color = colors[index % len(colors)]
        coords = " ".join(
            f'{scale_x(value["minute"]):.2f},{scale_y(value["indexed"]):.2f}'
            for value in item["values"]
        )
        symbol = escape(item["symbol"])
        last = item["values"][-1]
        label_x = margin["left"] + inner_width + 10
        label_y = scale_y(last["indexed"])
        parts.append(f'<polyline points="{coords}" fill="none" stroke="{color}" stroke-width="2" />')
        parts.append(f'<circle cx="{scale_x(last["minute"]):.2f}" cy="{label_y:.2f}" r="3" fill="{color}" />')
        parts.append(f'<text x="{label_x}" y="{label_y + 4:.2f}" font-size="11" fill="{color}">{symbol}</text>')

    parts.append("</svg>")
    return HTML("".join(parts))


display(render_price_paths(series))

In [4]:
summary = []
for item in series:
    values = item["values"]
    start = values[0]["price"]
    end = values[-1]["price"]
    summary.append(
        {
            "symbol": item["symbol"],
            "start_price": round(start, 4),
            "end_price": round(end, 4),
            "percent_change": round((end / start - 1) * 100, 3),
        }
    )

summary

[{'symbol': 'AAPL',
  'start_price': 224.1981,
  'end_price': 223.6886,
  'percent_change': -0.227},
 {'symbol': 'MSFT',
  'start_price': 429.815,
  'end_price': 430.5733,
  'percent_change': 0.176},
 {'symbol': 'NVDA',
  'start_price': 901.12,
  'end_price': 904.4597,
  'percent_change': 0.371},
 {'symbol': 'AMZN',
  'start_price': 185.517,
  'end_price': 183.8727,
  'percent_change': -0.886},
 {'symbol': 'GOOGL',
  'start_price': 165.4322,
  'end_price': 162.905,
  'percent_change': -1.528},
 {'symbol': 'META',
  'start_price': 509.6798,
  'end_price': 510.9315,
  'percent_change': 0.246},
 {'symbol': 'TSLA',
  'start_price': 250.0559,
  'end_price': 253.1053,
  'percent_change': 1.219},
 {'symbol': 'JPM',
  'start_price': 210.212,
  'end_price': 211.2985,
  'percent_change': 0.517},
 {'symbol': 'XOM',
  'start_price': 118.7985,
  'end_price': 118.5557,
  'percent_change': -0.204},
 {'symbol': 'UNH',
  'start_price': 575.3029,
  'end_price': 570.9879,
  'percent_change': -0.75}]